In [ ]:
pip install snowflake-snowpark-python sentence-transformers google-generativeai


INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of proto-plus to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 16.7 MB/s  0:00:

In [ ]:
import json
from sentence_transformers import SentenceTransformer
from snowflake.snowpark import Session

# 1. Connect to Snowflake
connection_parameters = {
    "account": "",
    "user": "",
    "password": "",
    "warehouse": "",
    "database": "",
    "schema": "",
}
session = Session.builder.configs(connection_parameters).create()

# 2. Pull parsed logs from Snowflake into a Pandas DataFrame
df = session.table("PARSED_LOGS").to_pandas()

# 3. Generate Embeddings locally using open-source Hugging Face model (384 dimensions)
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["MESSAGE"].tolist())

# 4. Prepare DataFrame with JSON-serialized embedding strings
df["EMBEDDING_STR"] = [json.dumps(emb.tolist()) for emb in embeddings]

# Select only the columns needed for staging
staging_df = df[["LOG_ID", "MESSAGE", "EMBEDDING_STR"]].copy()
staging_df.columns = ["LOG_ID", "MESSAGE", "EMBEDDING_STR"]

# 5. Write dataframe to a temporary staging table in Snowflake
session.write_pandas(
    staging_df,
    "TEMP_EMBEDDING_STAGE",
    auto_create_table=True,
    overwrite=True,
)

# 6. Clear target table and bulk-insert using server-side PARSE_JSON
session.sql("TRUNCATE TABLE LOG_EMBEDDINGS_FREE").collect()

bulk_insert_query = """
    INSERT INTO LOG_EMBEDDINGS_FREE (log_id, message, embedding)
    SELECT LOG_ID, MESSAGE, PARSE_JSON(EMBEDDING_STR)
    FROM TEMP_EMBEDDING_STAGE
"""
session.sql(bulk_insert_query).collect()

# Clean up temporary staging table
session.sql("DROP TABLE IF EXISTS TEMP_EMBEDDING_STAGE").collect()

print(
    "Successfully generated local embeddings and bulk-synced back to"
    " Snowflake!"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully generated local embeddings and bulk-synced back to Snowflake!


In [ ]:
import json
from google import genai
from sentence_transformers import SentenceTransformer, util
from snowflake.snowpark import Session

# 1. Connect to Snowflake
connection_parameters = {
    "account": "",
    "user": "",
    "password": "",
    "warehouse": "",
    "database": "",
    "schema": "",
}
session = Session.builder.configs(connection_parameters).create()

# 2. Pull logs and embeddings from Snowflake
df = session.table("LOG_EMBEDDINGS_FREE").to_pandas()

# Convert stored JSON embedding strings back to Python lists of floats
import numpy as np

df["embedding_list"] = df["EMBEDDING"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

# 3. Initialize model & modern Gemini client
model = SentenceTransformer("all-MiniLM-L6-v2")
client = genai.Client(
    api_key="API-KEY here"
)

def run_sre_copilot(user_query: str):
  # Generate query embedding
  query_vector = model.encode(user_query, convert_to_tensor=True)

  # Compute cosine similarities against all stored log embeddings locally
  stored_embeddings = np.array(df["embedding_list"].tolist())
  stored_tensor = model.encode(
      df["MESSAGE"].tolist(), convert_to_tensor=True
  )

  cos_scores = util.cos_sim(query_vector, stored_tensor)[0]
  top_results = torch.topk(cos_scores, k=min(2, len(df)))

  # Extract relevant log messages
  context_logs = "\n".join(
      [df.iloc[idx.item()]["MESSAGE"] for idx in top_results.indices]
  )

  # Construct prompt for Gemini
  prompt = f"""
    You are an expert SRE Engineer. Analyze the issue using ONLY the log context below.

    Log Context:
    {context_logs}

    User Query: {user_query}
    """

  # Modern Gemini API call
  response = client.models.generate_content(
      model="gemini-3.8-flash",
      contents=prompt,
  )
  return response.text


# Test the pipeline
import torch

print(run_sre_copilot("Why did the payment system fail?"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Based on the provided log context, the payment system experienced failures due to two specific issues:

1. **Connection Timeout:** There was a connection timeout while communicating with the Stripe API endpoint.
2. **Insufficient Funds:** A transaction was declined because account ID #94821 had insufficient funds.
